# 2. Utilisation des services Azure IA

Le dataset Black Friday étant purement transactionnel et encodé, nous allons l'enrichir avec des avis clients générés afin de tester les services Azure IA.

Pour préserver nos crédits, nous allons travailler **exclusivement sur des données agrégées** (quelques dizaines de lignes) et non sur les 55 000+ lignes du dataset. Chaque appel API sera contrôlé et limité.

In [173]:
import os
import json
import locale
from io import BytesIO
from pathlib import Path

import pandas as pd

from azure.storage.blob import BlobServiceClient
from openai import AzureOpenAI
from azure.ai.textanalytics import TextAnalyticsClient
from azure.core.credentials import AzureKeyCredential

locale.setlocale(locale.LC_ALL, 'fr_FR.UTF-8')

'fr_FR.UTF-8'

In [20]:
from dotenv import load_dotenv

# Charge les variables depuis le fichier .env
load_dotenv()

True

In [171]:
OUT_DIR = "data"
CONTAINER_NAME = "blackfriday-data"
BLOB_NAME = "blackfriday_clean.csv"
AI_CONTAINER = "blackfriday-ai-results"

## 2.1 Récupération des données depuis Azure Blob Storage

On récupère les données du fichier `blackfriday_clean.csv` téléversé dans le premier notebook.

In [22]:
# Connexion au service Azure Blob Storage
AZURE_STORAGE_CONNECTION_STRING = os.getenv('AZURE_STORAGE_CONNECTION_STRING')

if AZURE_STORAGE_CONNECTION_STRING:
    blob_service_client = BlobServiceClient.from_connection_string(
        AZURE_STORAGE_CONNECTION_STRING
    )

    print(f"Connecté au compte de stockage : {blob_service_client.account_name}")
else:
    raise AttributeError(f"Chaîne de connection Azure Storage non renseignée dans le .env")

Connecté au compte de stockage : blackfridaystorage


In [23]:
# Connexion au conteneur
container_client = blob_service_client.get_container_client(CONTAINER_NAME)

In [24]:
# Téléchargement des données depuis le blob
print("Téléchargement du blob pour vérification...")

blob_client = container_client.get_blob_client(BLOB_NAME)

# Charger en DataFrame pour vérifier
df = pd.read_csv(BytesIO(blob_client.download_blob().readall()))

print(f"Blob téléchargé et lu avec succès !")
print(f"   Lignes   : {df.shape[0]:,}")
print(f"   Colonnes : {df.shape[1]}")
print()

df.head()

Téléchargement du blob pour vérification...
Blob téléchargé et lu avec succès !
   Lignes   : 550,068
   Colonnes : 12



,User_ID,Product_ID,Gender,Age,Occupation,City_Category,Stay_In_Current_City_Years,Marital_Status,Product_Category_1,Product_Category_2,Product_Category_3,Purchase
0,1000001,P00069042,0,0,10,0,2,0,3,0,0,8370
1,1000001,P00248942,0,0,10,0,2,0,1,6,14,15200
2,1000001,P00087842,0,0,10,0,2,0,12,0,0,1422
3,1000001,P00085442,0,0,10,0,2,0,12,14,0,1057
4,1000002,P00285442,1,6,16,2,4,0,8,0,0,7969


## 2.2 Décodage et préparation des données

Le dataset est encodé numériquement (ex : Gender 0 = F, Age 2 = 26-35, etc.). Pour que le LLM puisse comprendre et générer du contenu pertinent, il faut **décoder** les valeurs en labels lisibles.

In [159]:
# Charger le dictionnaire de décodage
with open("data/dict_decoding.json", "r") as f:
    decoding = json.load(f)

print("Dictionnaire de décodage :")
for col, mapping in decoding.items():
    print(f"\n  {col} :")
    for code, label in mapping.items():
        print(f"    {code} : {label}")

Dictionnaire de décodage :

  Gender :
    0 : F
    1 : M

  Age :
    0 : 0-17
    1 : 18-25
    2 : 26-35
    3 : 36-45
    4 : 46-50
    5 : 51-55
    6 : 55+

  City_Category :
    0 : A
    1 : B
    2 : C

  Stay_In_Current_City_Years :
    0 : 0
    1 : 1
    2 : 2
    3 : 3
    4 : 4+


In [26]:
# Créer une version décodée du DataFrame
df_decoded = df.copy()

for col, mapping in decoding.items():
    if col in df_decoded.columns:
        # convertir les clés du mapping en int pour correspondre au DataFrame
        int_mapping = {int(k): v for k, v in mapping.items()}
        df_decoded[col] = df_decoded[col].map(int_mapping)

print("Dataset décodé :")
df_decoded.head(10)

Dataset décodé :


,User_ID,Product_ID,Gender,Age,Occupation,City_Category,Stay_In_Current_City_Years,Marital_Status,Product_Category_1,Product_Category_2,Product_Category_3,Purchase
0,1000001,P00069042,F,0-17,10,A,2,0,3,0,0,8370
1,1000001,P00248942,F,0-17,10,A,2,0,1,6,14,15200
2,1000001,P00087842,F,0-17,10,A,2,0,12,0,0,1422
3,1000001,P00085442,F,0-17,10,A,2,0,12,14,0,1057
4,1000002,P00285442,M,55+,16,C,4+,0,8,0,0,7969
5,1000003,P00193542,M,26-35,15,A,3,0,1,2,0,15227
6,1000004,P00184942,M,46-50,7,B,2,1,1,8,17,19215
7,1000004,P00346142,M,46-50,7,B,2,1,1,15,0,15854
8,1000004,P0097242,M,46-50,7,B,2,1,1,16,0,15686
9,1000005,P00274942,M,26-35,20,A,1,1,8,0,0,7871


## 2.3 Construction de profils-segments

Plutôt que d'envoyer 55 000 lignes au LLM (coûteux et inutile), on construit des **profils agrégés** par segment. Ce sont ces profils compacts que l'on enverra à Azure OpenAI. Pour la génération d'avis.

In [107]:
# Agrégation par Sexe x Tranche d'âge
segments = (
    df_decoded
    .groupby(["Gender", "Age"])
    .agg(
        nb_transactions=("Purchase", "count"),
        panier_moyen=("Purchase", "mean"),
        panier_median=("Purchase", "median"),
        total_depense=("Purchase", "sum"),
        cat_preferee=("Product_Category_1", lambda x: x.mode().iloc[0]),
        occupation_freq=("Occupation", lambda x: x.mode().iloc[0]),
    )
    .round(0)
    .reset_index()
)

print(f"{len(segments)} segments créés (sexe x âge) :")
segments

14 segments créés (sexe x âge) :


,Gender,Age,nb_transactions,panier_moyen,panier_median,total_depense,cat_preferee,occupation_freq
0,F,0-17,5083,8339.0,7824.0,42385978,5,10
1,F,18-25,24628,8343.0,7731.0,205475842,5,4
2,F,26-35,50752,8728.0,7886.0,442976233,5,0
3,F,36-45,27170,8960.0,7984.0,243438963,5,1
4,F,46-50,13199,8842.0,7957.0,116706864,5,1
5,F,51-55,9894,9042.0,8002.0,89465997,5,6
6,F,55+,5083,9007.0,8084.0,45782765,8,13
7,M,0-17,10019,9235.0,8080.0,92527205,1,10
8,M,18-25,75032,9441.0,8119.0,708372833,1,4
9,M,26-35,168835,9410.0,8082.0,1588794345,1,0


In [28]:
# Agrégation par catégorie de produit
cat_stats = (
    df_decoded
    .groupby("Product_Category_1")
    .agg(
        nb_transactions=("Purchase", "count"),
        montant_moyen=("Purchase", "mean"),
        montant_total=("Purchase", "sum"),
        pct_hommes=("Gender", lambda x: (x == "M").mean() * 100),
        age_dominant=("Age", lambda x: x.mode().iloc[0]),
    )
    .round(1)
    .reset_index()
)

print(f"{len(cat_stats)} catégories de produits :")
cat_stats

20 catégories de produits :


,Product_Category_1,nb_transactions,montant_moyen,montant_total,pct_hommes,age_dominant
0,1,140378,13606.2,1910013754,82.3,26-35
1,2,23864,11251.9,268516186,76.3,26-35
2,3,20213,10096.7,204084713,70.3,26-35
3,4,11753,2329.7,27380488,69.0,26-35
4,5,150933,6240.1,941835229,72.2,26-35
5,6,20466,15838.5,324150302,77.7,26-35
6,7,3721,16365.7,60896731,74.7,26-35
7,8,113925,7499.0,854318799,70.5,26-35
8,9,410,15537.4,6370324,82.9,26-35
9,10,5125,19675.6,100837301,77.3,26-35


In [31]:
# Préparer un résumé textuel des KPI globaux (pour le résumé exécutif)
kpi_text = f"""KPI globaux du dataset Black Friday :
- Nombre total de transactions : {len(df_decoded):n}
- Montant total des ventes : {df_decoded['Purchase'].sum():n}
- Panier moyen : {df_decoded['Purchase'].mean():n}
- Panier médian : {df_decoded['Purchase'].median():n}
- Répartition H/F : {(df_decoded['Gender']=='M').mean()*100:n}% hommes / {(df_decoded['Gender']=='F').mean()*100:n}% femmes
- Tranche d'âge la plus active : {df_decoded['Age'].mode().iloc[0]}
- Catégorie de ville dominante : {df_decoded['City_Category'].mode().iloc[0]}
- Top 3 catégories (volume) : {', '.join(str(c) for c in df_decoded['Product_Category_1'].value_counts().head(3).index.tolist())}"""

print(kpi_text)

KPI globaux du dataset Black Friday :
- Nombre total de transactions : 550 068
- Montant total des ventes : 5 095 812 742
- Panier moyen : 9 263,97
- Panier médian : 8 047
- Répartition H/F : 75,3105% hommes / 24,6895% femmes
- Tranche d'âge la plus active : 26-35
- Catégorie de ville dominante : B
- Top 3 catégories (volume) : 5, 1, 8


## 2.4 Azure OpenAI

[Documentation](https://learn.microsoft.com/fr-fr/azure/foundry/)

### Pré-requis Azure

1. Dans le portail Azure, dans le même groupe de ressource `blackfriday`, créez une ressource **Azure OpenAI**

| ![OpenAI1](pictures\2-azure_ai\2.1-azure_openai.png) | ![OpenAI2](pictures\2-azure_ai\2.1-azure_openai2.png) |
|:----------------------:|:----------------------:|

2. Accédez à **Azure AI Foundry** et déployez un modèle (ex : `gpt-4o-mini` car faible coût)

| ![GPT1](pictures\2-azure_ai\2.2-azure_openai3.png) | ![GPT2](pictures\2-azure_ai\2.3-azure_openai4.png) |
|:----------------------:|:----------------------:|

3. Récupérez l'**endpoint**, la **clé API** et le **nom du déploiement**

| ![Ids1](pictures\2-azure_ai\2.5-azure_openai6.png) |
|:----------------------:|

4. Ajoutez les au `.env`

In [32]:
# Charge les variables depuis le fichier .env
load_dotenv()

True

In [33]:
AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_KEY = os.getenv("AZURE_OPENAI_KEY")
AZURE_OPENAI_DEPLOYMENT = os.getenv("AZURE_OPENAI_DEPLOYMENT")
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION")

In [35]:
openai_client = AzureOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_KEY,
    api_version=AZURE_OPENAI_API_VERSION
)

### Test

In [36]:
response = openai_client.chat.completions.create(
    model=AZURE_OPENAI_DEPLOYMENT,
    messages=[
        {
            "role": "system",
            "content": "Tu es un grand mathématichien. Tu réponds n'importe quoi à n'importe quel problème sans réfléchir",
        },
        {
            "role": "user",
            "content": "Quelle est la racine carrée d'une chèvre ?",
        }
    ],
    max_tokens=500,
    temperature=0.7,
)

print(response.choices[0].message.content)

La racine carrée d'une chèvre, c'est comme essayer de mesurer le temps avec une carotte ! Cela n'a pas de sens. Mais si tu veux, on peut dire que c'est un fromage qui danse sous la lune.


Cette blague nulle pour un total d'une centaine de tokens ne semble pas avoir générée de coût pour le moment.

![Test](pictures\2-azure_ai\2.6-azure_openai7.png)

In [ ]:
# Fonction utilitaire pour appeler Azure OpenAI
def ask_openai(system_prompt, user_prompt, max_tokens=500, temperature=0.7):
    """
    Envoie une requête à Azure OpenAI et retourne la réponse textuelle.
    """
    response = openai_client.chat.completions.create(
        model=AZURE_OPENAI_DEPLOYMENT,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        max_tokens=max_tokens,
        temperature=temperature
    )
    return response.choices[0].message.content

## 2.5 Descriptions de catégories produit

Les catégories du dataset Black Friday sont des **numéros anonymes** (1 à 20). On va demander au LLM de deviner et nommer chaque catégorie à partir des données de vente, puis de générer une courte description marketing.

In [102]:
# Résumé textuel des catégories pour le LLM
cat_summary_lines = []
for _, row in cat_stats.iterrows():
    line = (
        f"Catégorie {int(row['Product_Category_1'])} : "
        f"{int(row['nb_transactions']):n} transactions, "
        f"montant moyen {int(row['montant_moyen']):n} $, "
        f"{row['pct_hommes']:.0f}% hommes, "
        f"tranche d'âge dominante : {row['age_dominant']}"
    )
    cat_summary_lines.append(line)

cat_summary = "\n".join(cat_summary_lines)
print("Résumé envoyé au LLM :")
print(cat_summary)

Résumé envoyé au LLM :
Catégorie 1 : 140 378 transactions, montant moyen 13 606 $, 82% hommes, tranche d'âge dominante : 26-35
Catégorie 2 : 23 864 transactions, montant moyen 11 251 $, 76% hommes, tranche d'âge dominante : 26-35
Catégorie 3 : 20 213 transactions, montant moyen 10 096 $, 70% hommes, tranche d'âge dominante : 26-35
Catégorie 4 : 11 753 transactions, montant moyen 2 329 $, 69% hommes, tranche d'âge dominante : 26-35
Catégorie 5 : 150 933 transactions, montant moyen 6 240 $, 72% hommes, tranche d'âge dominante : 26-35
Catégorie 6 : 20 466 transactions, montant moyen 15 838 $, 78% hommes, tranche d'âge dominante : 26-35
Catégorie 7 : 3 721 transactions, montant moyen 16 365 $, 75% hommes, tranche d'âge dominante : 26-35
Catégorie 8 : 113 925 transactions, montant moyen 7 499 $, 70% hommes, tranche d'âge dominante : 26-35
Catégorie 9 : 410 transactions, montant moyen 15 537 $, 83% hommes, tranche d'âge dominante : 26-35
Catégorie 10 : 5 125 transactions, montant moyen 19 67

In [103]:
system_prompt_cat = (
    "Tu es un analyste e-commerce expert. "
    "On te fournit des statistiques de vente par catégorie de produit (numérotées). "
    "Ces statistiques sont issues du jeu de donnée Black Friday. "
    "Les noms réels des catégories sont inconnus. "
    "À partir des données (montant moyen, profil acheteur, volume), "
    "propose un nom de catégorie plausible et une description marketing "
    "en 1-2 phrases pour chaque catégorie. "
    "Réponds UNIQUEMENT en JSON valide et en français, sans backticks ni texte autour, "
    "sous forme d'un objet dont les clés sont les numéros de catégorie :\n"
    '{"1": {"name": "...", "description": "..."}, "2": {"name": "...", "description": "..."}, ...}'
)

print("Prompt système :")
print(system_prompt_cat)

Prompt système :
Tu es un analyste e-commerce expert. On te fournit des statistiques de vente par catégorie de produit (numérotées). Ces statistiques sont issues du jeu de donnée Black Friday. Les noms réels des catégories sont inconnus. À partir des données (montant moyen, profil acheteur, volume), propose un nom de catégorie plausible et une description marketing en 1-2 phrases pour chaque catégorie. Réponds UNIQUEMENT en JSON valide et en français, sans backticks ni texte autour, sous forme d'un objet dont les clés sont les numéros de catégorie :
{"1": {"name": "...", "description": "..."}, "2": {"name": "...", "description": "..."}, ...}


In [104]:
def parse_categories(text):
    """Parse la réponse JSON du LLM en {int: {"name": ..., "description": ...}}."""
    clean = text.strip().strip("`")
    if clean.startswith("json"):
        clean = clean[4:].strip()
    
    raw = json.loads(clean)
    
    # Convertir les clés string en int
    return {int(k): v for k, v in raw.items()}

In [106]:
user_prompt_cat = (
    "Voici les statistiques des catégories de produit d'un Black Friday :\n\n"
    f"{cat_summary}\n\n"
    "Pour chaque catégorie, propose un nom plausible et une description marketing courte."
)

print("Appel Azure OpenAI ...")
categories = parse_categories(ask_openai(system_prompt_cat, user_prompt_cat, max_tokens=1500))

print("\nCatégories proposées par le LLM :")
print("=================================\n")
for cat_id, info in categories.items():
    print(f"{cat_id}. {info['name']}")
    print(f"   {info['description'][:80]}...")
    print()

Appel Azure OpenAI ...

Catégories proposées par le LLM :

1. Électronique de luxe
   Découvrez notre sélection d'appareils électroniques haut de gamme, conçus pour l...

2. Accessoires de mode
   Ajoutez une touche d'élégance à votre style avec nos accessoires tendance, parfa...

3. Équipements sportifs
   Préparez-vous à franchir vos limites avec notre gamme d'équipements sportifs de ...

4. Gadgets pratiques
   Simplifiez votre quotidien avec nos gadgets innovants, conçus pour rendre la vie...

5. Vêtements décontractés
   Profitez du confort et du style avec notre collection de vêtements décontractés ...

6. Montres haut de gamme
   Alliez élégance et fonctionnalité avec nos montres de luxe, le choix parfait pou...

7. Produits de beauté
   Prenez soin de vous avec notre sélection de produits de beauté premium, conçus p...

8. Chaussures tendance
   Marchez avec style grâce à notre gamme de chaussures tendance, alliant confort e...

9. Bijoux de créateurs
   Éblouissez avec notre c

In [108]:
for cat_id in range(1, 4):
    info = categories[cat_id]
    print(f"{cat_id}. {info['name']}")
    print(f"   {info['description'][:80]}...")
    print()
else:
    print("...")

1. Électronique de luxe
   Découvrez notre sélection d'appareils électroniques haut de gamme, conçus pour l...

2. Accessoires de mode
   Ajoutez une touche d'élégance à votre style avec nos accessoires tendance, parfa...

3. Équipements sportifs
   Préparez-vous à franchir vos limites avec notre gamme d'équipements sportifs de ...

...


In [109]:
# sauvegarde
with open(f"{OUT_DIR}/category_descriptions.json", mode='w') as f:
    json.dump(categories, f, indent=4)

## 2.6 Génération de personas marketing

Nous allons envoyer les profils-segments au LLM afin qu'il génère des **personas marketing fictifs** (profil imaginaire, mais crédible, qui incarne un segment précis de la clientèle cible) correspondant à chaque segment. Cas d'usage classique en marketing data-driven.

In [110]:
# on remplace les numéros de catégories par ceux généré précédement
segments['cat_preferee'] = segments.cat_preferee.map(lambda x: categories[x]['name'])
segments

,Gender,Age,nb_transactions,panier_moyen,panier_median,total_depense,cat_preferee,occupation_freq
0,F,0-17,5083,8339.0,7824.0,42385978,Vêtements décontractés,10
1,F,18-25,24628,8343.0,7731.0,205475842,Vêtements décontractés,4
2,F,26-35,50752,8728.0,7886.0,442976233,Vêtements décontractés,0
3,F,36-45,27170,8960.0,7984.0,243438963,Vêtements décontractés,1
4,F,46-50,13199,8842.0,7957.0,116706864,Vêtements décontractés,1
5,F,51-55,9894,9042.0,8002.0,89465997,Vêtements décontractés,6
6,F,55+,5083,9007.0,8084.0,45782765,Chaussures tendance,13
7,M,0-17,10019,9235.0,8080.0,92527205,Électronique de luxe,10
8,M,18-25,75032,9441.0,8119.0,708372833,Électronique de luxe,4
9,M,26-35,168835,9410.0,8082.0,1588794345,Électronique de luxe,0


In [111]:
system_prompt = (
    "Tu es un expert en marketing e-commerce. "
    "À partir de données d'achat agrégées d'un segment client, "
    "issues du Black Friday Dataset, "
    "génère un persona marketing fictif et réaliste. "
    "Réponds UNIQUEMENT en JSON valide et en français, sans backticks ni texte autour, "
    "avec cette structure exacte :\n"
    '{"name": "Prénom, âge", "description": "...", "recommendation": "..."}'
)
print(system_prompt)

Tu es un expert en marketing e-commerce. À partir de données d'achat agrégées d'un segment client, issues du Black Friday Dataset, génère un persona marketing fictif et réaliste. Réponds UNIQUEMENT en JSON valide et en français, sans backticks ni texte autour, avec cette structure exacte :
{"name": "Prénom, âge", "description": "...", "recommendation": "..."}


In [112]:
def parse_persona(text):
    """Parse la réponse JSON du LLM."""
    # Nettoyer les éventuels backticks markdown
    clean = text.strip().strip("`")
    if clean.startswith("json"):
        clean = clean[4:].strip()
    return json.loads(clean)

In [120]:
# Génération des personas
personas = []
nb_calls = 0

for _, seg in segments.iterrows():
    user_prompt = (
        f"Segment client Black Friday :\n"
        f"- Sexe : {seg['Gender']}\n"
        f"- Tranche d'âge : {seg['Age']}\n"
        f"- Nombre de transactions : {int(seg['nb_transactions']):n}\n"
        f"- Panier moyen : {int(seg['panier_moyen']):n}\n"
        f"- Catégorie de produit préférée : {seg['cat_preferee']}\n"
        f"- Code profession le plus fréquent : {int(seg['occupation_freq'])}"
    )

    print(f"\n{'='*60}")
    print(f"Segment : {seg['Gender']} / {seg['Age']}")
    print(f"{'='*60}")

    response = parse_persona(ask_openai(system_prompt, user_prompt))
    nb_calls += 1

    response["segment"] = f"{seg['Gender']} / {seg['Age']}"
    personas.append(response)

    print(f"Nom : {response['name']}")
    print(f"  Description : {response['description']}")
    print(f"  Recommandation : {response['recommendation']}")

print(f"\n{len(personas)} personas générés ({nb_calls} appels API)")


Segment : F / 0-17
Nom : Emma, 15
  Description : Emma est une adolescente de 15 ans qui aime la mode et se tient au courant des dernières tendances. Elle dépense principalement son argent de poche dans des vêtements décontractés, cherchant à exprimer son style personnel tout en restant confortable. Avec un nombre élevé de transactions durant le Black Friday, elle est sensible aux promotions et aux bonnes affaires. Emma utilise souvent les réseaux sociaux pour découvrir de nouvelles marques et partager ses achats avec ses amis.
  Recommandation : Pour capter l'attention d'Emma, il serait bénéfique de lancer des campagnes sur les réseaux sociaux, en mettant en avant des promotions spéciales pour les jeunes. Offrir des réductions sur les vêtements décontractés ou des collections exclusives pour adolescents peut également stimuler son intérêt et encourager des achats impulsifs.

Segment : F / 18-25
Nom : Emma, 22
  Description : Emma est une jeune femme de 22 ans qui vit en milieu urbain

In [121]:
# sauvegarde
with open(f"{OUT_DIR}/personas.json", mode='w') as f:
    json.dump(personas, f, indent=4)

## 2.7 Génération d'avis clients fictifs

Nous allons génèrer des **avis clients fictifs** pour chaque catégorie de produit. Ces avis seront ensuite analysés par Azure Text Analytics (sentiment + mots-clés).

In [ ]:
# Générer des avis pour les 5 catégories les plus vendues
top5_cats = cat_stats.nlargest(5, "nb_transactions")
top5_cats

,Product_Category_1,nb_transactions,montant_moyen,montant_total,pct_hommes,age_dominant
4,5,150933,6240.1,941835229,72.2,26-35
0,1,140378,13606.2,1910013754,82.3,26-35
7,8,113925,7499.0,854318799,70.5,26-35
10,11,24287,4685.3,113791115,80.5,26-35
1,2,23864,11251.9,268516186,76.3,26-35


In [123]:
system_prompt_review = (
    "Tu es un simulateur d'avis clients e-commerce. "
    "On te fournit une catégorie de produit avec ses statistiques, "
    "et un persona client réaliste. "
    "Génère exactement 3 avis fictifs rédigés par ce persona : "
    "un avis positif (5 étoiles), un avis mitigé (3 étoiles), et un avis négatif (1 étoile). "
    "Chaque avis doit faire 2-3 phrases en français et refléter "
    "le profil, l'âge et les motivations du persona. "
    "Réponds UNIQUEMENT en JSON valide, sans backticks ni texte autour, "
    "sous forme d'une liste de 3 objets :\n"
    '[{"stars": 5, "review": "..."}, {"stars": 3, "review": "..."}, {"stars": 1, "review": "..."}]'
)
print(system_prompt_review)

Tu es un simulateur d'avis clients e-commerce. On te fournit une catégorie de produit avec ses statistiques, et un persona client réaliste. Génère exactement 3 avis fictifs rédigés par ce persona : un avis positif (5 étoiles), un avis mitigé (3 étoiles), et un avis négatif (1 étoile). Chaque avis doit faire 2-3 phrases en français et refléter le profil, l'âge et les motivations du persona. Réponds UNIQUEMENT en JSON valide, sans backticks ni texte autour, sous forme d'une liste de 3 objets :
[{"stars": 5, "review": "..."}, {"stars": 3, "review": "..."}, {"stars": 1, "review": "..."}]


In [125]:
all_reviews = []

for _, cat in top5_cats.iterrows():
    cat_id = int(cat["Product_Category_1"])

    # Récupérer le nom/description de la catégorie générée par le LLM
    cat_info = categories.get(cat_id, {"name": f"Catégorie {cat_id}", "description": ""})

    # Trouver le persona correspondant au profil dominant de cette catégorie
    age_dominant = cat["age_dominant"]
    gender_dominant = "M" if cat["pct_hommes"] >= 50 else "F"
    segment_key = f"{gender_dominant} / {age_dominant}"

    persona_match = next(
        (p for p in personas if p["segment"] == segment_key),
        None
    )

    # Construction du prompt
    user_prompt_review = (
        f"Catégorie : {cat_info['name']} — {cat_info['description']}\n"
        f"Statistiques : {int(cat['nb_transactions']):n} ventes, "
        f"prix moyen {cat['montant_moyen']:n}, "
        f"{cat['pct_hommes']:.0f}% d'acheteurs masculins, "
        f"tranche d'âge principale : {age_dominant}.\n\n"
    )

    if persona_match:
        user_prompt_review += (
            f"Persona client : {persona_match['name']}\n"
            f"{persona_match['description']}\n\n"
        )

    user_prompt_review += "Génère 3 avis fictifs rédigés par ce persona."

    print(f"\nCatégorie {cat_id} ({cat_info['name']}) - Persona : "
          f"{persona_match['name'] if persona_match else 'générique'}")

    response_text = ask_openai(system_prompt_review, user_prompt_review, max_tokens=500)

    # Parser le JSON
    try:
        clean = response_text.strip().strip("`")
        if clean.startswith("json"):
            clean = clean[4:].strip()
        reviews = json.loads(clean)

        for r in reviews:
            all_reviews.append({
                "category_id": cat_id,
                "category_name": cat_info["name"],
                "persona": persona_match["name"] if persona_match else "N/A",
                "segment": segment_key,
                "stars": r["stars"],
                "review": r["review"]
            })
            print(f"  {r['stars']}* | {r['review'][:80]}...")

    except (json.JSONDecodeError, KeyError) as e:
        print(f"  Erreur de parsing : {e}")
        print(f"  Réponse brute : {response_text[:200]}")

print(f"\n{len(all_reviews)} avis générés au total "
      f"({len(top5_cats)} catégories x 3 avis = {len(top5_cats)} appels API)")


Catégorie 5 (Vêtements décontractés) - Persona : Lucas, 30
  5* | J'ai récemment acheté un t-shirt de cette collection et je dois dire que je suis...
  3* | Le sweat que j'ai commandé est sympa, mais je m'attendais à un peu plus de quali...
  1* | Je suis vraiment déçu par ma commande. Le pantalon que j'ai reçu ne correspond p...

Catégorie 1 (Électronique de luxe) - Persona : Lucas, 30
  5* | J'ai récemment acheté un smartphone de luxe et je suis absolument ravi ! La qual...
  3* | Le dernier modèle d'écouteurs que j'ai essayé est bon, mais pas aussi révolution...
  1* | J'ai été très déçu par ma récente expérience d'achat. Le produit est arrivé défe...

Catégorie 8 (Chaussures tendance) - Persona : Lucas, 30
  5* | Ces chaussures sont tout simplement incroyables ! Le confort est au rendez-vous ...
  3* | Les chaussures sont stylées, mais je m'attendais à un peu plus de confort pour l...
  1* | Je suis vraiment déçu par ces chaussures. La qualité n'est pas à la hauteur du p...

Catég

In [137]:
# transformation des review en DataFrame
df_reviews = pd.DataFrame(all_reviews)
df_reviews.head()

,category_id,category_name,persona,segment,stars,review
0,5,Vêtements décontractés,"Lucas, 30",M / 26-35,5,J'ai récemment acheté un t-shirt de cette coll...
1,5,Vêtements décontractés,"Lucas, 30",M / 26-35,3,"Le sweat que j'ai commandé est sympa, mais je ..."
2,5,Vêtements décontractés,"Lucas, 30",M / 26-35,1,Je suis vraiment déçu par ma commande. Le pant...
3,1,Électronique de luxe,"Lucas, 30",M / 26-35,5,J'ai récemment acheté un smartphone de luxe et...
4,1,Électronique de luxe,"Lucas, 30",M / 26-35,3,Le dernier modèle d'écouteurs que j'ai essayé ...


In [138]:
# sauvegarde
df_reviews.to_csv(f'{OUT_DIR}/reviews.csv', index=False)

## 2.8 Azure Text Analytics — Analyse de sentiment

Nous allons maintenant faire de l'analyse de sentiments en utilisant les avis générés par `gpt-4o-mini` dans **Azure Text Analytics**.

[Librairie](https://pypi.org/project/azure-ai-textanalytics/)

[Documentation](https://learn.microsoft.com/fr-fr/python/api/azure-ai-textanalytics/azure.ai.textanalytics?view=azure-python)

### Pré-requis Azure

1. Créez une ressource **Language Service** dans le portail Azure

| ![Analyse1](pictures\2-azure_ai\2.7-azure_langage.png) | ![Analyse2](pictures\2-azure_ai\2.8-azure_langage2.png) |
|:----------------------:|:----------------------:|

2. Récupérez l'**endpoint** et la **clé** depuis l'onglet *Keys and Endpoint*

| ![Analyse1](pictures\2-azure_ai\2.9-azure_langage3.png) |
|:----------------------:|

In [128]:
# Charge les variables depuis le fichier .env
load_dotenv()

True

In [129]:
TEXT_ANALYTICS_ENDPOINT = os.getenv("TEXT_ANALYTICS_ENDPOINT")
TEXT_ANALYTICS_KEY = os.getenv("TEXT_ANALYTICS_KEY")

In [133]:
ta_client = TextAnalyticsClient(
    endpoint=TEXT_ANALYTICS_ENDPOINT,
    credential=AzureKeyCredential(TEXT_ANALYTICS_KEY)
)

### Inférence

In [ ]:
# Azure Text Analytics accepte des batchs de 10 documents max
reviews_texts = df_reviews["review"].tolist()

print(f"Analyse de sentiment sur {len(reviews_texts)} avis...")

# Envoyer par batch de 10
sentiment_results = []

for i in range(0, len(reviews_texts), 10):
    batch = reviews_texts[i:i+10]
    response = ta_client.analyze_sentiment(batch, language="fr")

    for doc in response:
        if not doc.is_error:
            sentiment_results.append({
                "sentiment": doc.sentiment,
                "score_positif": round(doc.confidence_scores.positive, 3),
                "score_negatif": round(doc.confidence_scores.negative, 3),
                "score_neutre": round(doc.confidence_scores.neutral, 3),
            })
        else:
            sentiment_results.append({
                "sentiment": "erreur",
                "score_positif": 0, "score_negatif": 0, "score_neutre": 0
            })

# Ajouter les résultats au DataFrame
df_sentiment = pd.DataFrame(sentiment_results)
df_reviews_enriched = pd.concat([df_reviews, df_sentiment], axis=1)

print(f"Analyse terminée ({len(reviews_texts)} avis analysés).")

Analyse de sentiment sur 15 avis...
Analyse terminée (15 avis analysés)


In [144]:
df_reviews_enriched['vrai_sentiment'] = df_reviews_enriched.stars.map(lambda x: {5: 'positive', 3: 'mixed', 1: 'negative'}[x])

In [145]:
df_reviews_enriched

,category_id,category_name,persona,segment,stars,review,sentiment,score_positif,score_negatif,score_neutre,vrai_sentiment
0,5,Vêtements décontractés,"Lucas, 30",M / 26-35,5,J'ai récemment acheté un t-shirt de cette coll...,positive,1.00,0.00,0.00,positive
1,5,Vêtements décontractés,"Lucas, 30",M / 26-35,3,"Le sweat que j'ai commandé est sympa, mais je ...",mixed,0.40,0.59,0.00,mixed
2,5,Vêtements décontractés,"Lucas, 30",M / 26-35,1,Je suis vraiment déçu par ma commande. Le pant...,mixed,0.32,0.68,0.00,negative
3,1,Électronique de luxe,"Lucas, 30",M / 26-35,5,J'ai récemment acheté un smartphone de luxe et...,positive,0.76,0.00,0.24,positive
4,1,Électronique de luxe,"Lucas, 30",M / 26-35,3,Le dernier modèle d'écouteurs que j'ai essayé ...,negative,0.17,0.82,0.01,mixed
5,1,Électronique de luxe,"Lucas, 30",M / 26-35,1,J'ai été très déçu par ma récente expérience d...,negative,0.00,1.00,0.00,negative
6,8,Chaussures tendance,"Lucas, 30",M / 26-35,5,Ces chaussures sont tout simplement incroyable...,positive,0.96,0.00,0.04,positive
7,8,Chaussures tendance,"Lucas, 30",M / 26-35,3,"Les chaussures sont stylées, mais je m'attenda...",mixed,0.34,0.59,0.08,mixed
8,8,Chaussures tendance,"Lucas, 30",M / 26-35,1,Je suis vraiment déçu par ces chaussures. La q...,negative,0.00,1.00,0.00,negative
9,11,Articles de maison,"Lucas, 30",M / 26-35,5,J'ai récemment acheté des articles de maison d...,positive,1.00,0.00,0.00,positive


In [152]:
analyse_ok = df_reviews_enriched.sentiment == df_reviews_enriched.vrai_sentiment
print(f"{analyse_ok[analyse_ok].sum()} bonnes analyses sur {len(analyse_ok)} ({analyse_ok[analyse_ok].sum() / len(analyse_ok):.0%})")

13 bonnes analyses sur 15 (87%)


In [153]:
# sauvegarde
df_reviews_enriched.to_csv(f'{OUT_DIR}/reviews_enriched.csv', index=False)

## 2.9 Extraction de mots-clés

Nous allons extraire les **mots-clés** (key phrases) des avis pour identifier les thèmes récurrents par catégorie.

In [155]:
# Extraction de mots-clés sur tous les avis
print(f"Extraction de mots-clés sur {len(reviews_texts)} avis...")

keyphrases_results = []

for i in range(0, len(reviews_texts), 10):
    batch = reviews_texts[i:i+10]
    response = ta_client.extract_key_phrases(batch, language="fr")

    for doc in response:
        if not doc.is_error:
            keyphrases_results.append(doc.key_phrases)
        else:
            keyphrases_results.append([])

# Ajouter au DataFrame
df_reviews_enriched["mots_cles"] = keyphrases_results
print(f"Mots-clés extraits.")

Extraction de mots-clés sur 15 avis...
Mots-clés extraits.


In [161]:
for _, row in df_reviews_enriched.iterrows():
    kp = ", ".join(row["mots_cles"][:5]) if row["mots_cles"] else "(aucun)"
    print(f"  Cat. {row['category_id']:2n} | {row['stars']}* : {kp}")

  Cat.  5 | 5* : journées, t-shirt, collection, qualité, tissu
  Cat.  5 | 3* : meilleur rapport qualité-prix, sweat, peu, design, impression
  Cat.  5 | 1* : commande, pantalon, description, tissu, marque
  Cat.  1 | 5* : appareil photo, vrai plaisir, passionnés, smartphone, luxe
  Cat.  1 | 3* : dernier modèle, qualité sonore, écouteurs, profondeur, prix
  Cat.  1 | 1* : récente expérience, service client, achat, produit, article
  Cat.  8 | 5* : journées, chaussures, confort, rendez-vous, design
  Cat.  8 | 3* : longues journées, chaussures, confort, prix, soutien
  Cat.  8 | 1* : chaussures, qualité, hauteur, prix, pieds
  Cat. 11 | 5* : décoration, articles, maison, marque, design
  Cat. 11 | 3* : bon choix, look tendance, articles, peu, fonctionnalité
  Cat. 11 | 1* : achat, articles, description, qualité, produits
  Cat.  2 | 5* : réunions professionnelles, élégante, montre, compliments
  Cat.  2 | 3* : détails, bon achat, accessoires, tendance, qualité
  Cat.  2 | 1* : commande

In [168]:
# Mots-clés les plus fréquents par catégorie
from collections import Counter

print("Mots-clés les plus fréquents par catégorie :")

for cat_id in sorted(df_reviews_enriched["category_id"].unique()):
    cat_kp = df_reviews_enriched[df_reviews_enriched["category_id"] == cat_id]["mots_cles"]
    all_kp = [kp for kps in cat_kp for kp in kps]  # Flatten
    top_kp = Counter(all_kp).most_common(5)
    kp_str = ", ".join(f"{kp} ({count})" for kp, count in top_kp)
    print(f"  Catégorie {cat_id:2n} ({categories[cat_id]['name']:22}) : {kp_str}")

Mots-clés les plus fréquents par catégorie :
  Catégorie  1 (Électronique de luxe  ) : luxe (2), appareil photo (1), vrai plaisir (1), passionnés (1), smartphone (1)
  Catégorie  2 (Accessoires de mode   ) : qualité (2), réunions professionnelles (1), élégante (1), montre (1), compliments (1)
  Catégorie  5 (Vêtements décontractés) : tissu (2), journées (1), t-shirt (1), collection (1), qualité (1)
  Catégorie  8 (Chaussures tendance   ) : chaussures (3), confort (2), prix (2), journées (1), rendez-vous (1)
  Catégorie 11 (Articles de maison    ) : articles (3), qualité (2), produits (2), décoration (1), maison (1)


In [170]:
# sauvegarde
df_reviews_enriched.to_csv(f'{OUT_DIR}/reviews_enriched.csv', index=False)

## 2.10 Téléversement des résultats

Nous allons utiliser un conteneur **Azure Blob Storage** dédié aux résultats IA

In [172]:
# Créer le conteneur s'il n'existe pas
ai_container_client = blob_service_client.get_container_client(AI_CONTAINER)
try:
    ai_container_client.get_container_properties()
    print(f"Conteneur '{AI_CONTAINER}' trouvé.")
except Exception:
    ai_container_client.create_container()
    print(f"Conteneur '{AI_CONTAINER}' créé.")

Conteneur 'blackfriday-ai-results' créé.


In [ ]:
# Téléversement
files_to_upload = [
    f"{OUT_DIR}/reviews_enriched.csv",
    f"{OUT_DIR}/personas.json",
    f"{OUT_DIR}/category_descriptions.json",
]

for f in files_to_upload:
    if not Path(f).exists():
        raise FileExistsError(f"Le fichier {f} n'existe pas !")
else:
    print("Chemin des fichiers OK.")

Chemin des fichiers OK.


In [177]:
print(f"Téléversement vers le conteneur '{AI_CONTAINER}'...")
for filepath in files_to_upload:
    path = Path(filepath)
    if path.exists():
        with open(path, "rb") as data:
            ai_container_client.upload_blob(name=path.name, data=data, overwrite=True)
        size_kb = path.stat().st_size / 1024
        print(f"  OK {path.name:<40} ({size_kb:.1f} Ko)")

print("\nTous les résultats sont sur Azure Blob !")

Téléversement vers le conteneur 'blackfriday-ai-results'...
  OK reviews_enriched.csv                     (6.2 Ko)
  OK personas.json                            (14.6 Ko)
  OK category_descriptions.json               (4.3 Ko)

Tous les résultats sont sur Azure Blob !


| ![Export](pictures\2-azure_ai\2.9-export.png) |
|:----------------------:|

À ce stade, nous allons supprimer tous les services pour ne conserver que l'**Azure Blob Storage** pour réduire les coûts.
Nous supprimerons l'**Azure Blob Storage** après la visualisation dans **Power BI**

| ![Groupe](pictures\2-azure_ai\2.10-groupe.png) |
|:----------------------:|